In [1]:
import os, sys
from pathlib import Path
import pandas as pd
import subprocess

In [2]:
sys.path.append("../../../training_data")

In [3]:
from utils.utils import Cif

In [4]:
with open("../../../training_data/7.Extra_set/features.pkl", "rb") as f:
    extras_featuresd = pd.read_pickle(f)

len(extras_featuresd), extras_featuresd

(30,
 {'21du':     Residues                                                          \
           pdb label_entity_id label_asym_id label_seq_id auth_asym_id   
  0       21du               5             E          264            R   
  1       21du               5             E          265            R   
  2       21du               5             E          266            R   
  3       21du               5             E          267            R   
  4       21du               5             E          268            R   
  ..       ...             ...           ...          ...          ...   
  273     21du               5             E          546            R   
  274     21du               5             E          547            R   
  275     21du               5             E          548            R   
  276     21du               5             E          549            R   
  277     21du               5             E          550            R   
  
                      

In [5]:
remaining = []

for pdb, feats in extras_featuresd.items():
    outdir = Path(pdb)
    if not outdir.exists():
        remaining.append(pdb)

len(remaining), remaining

(11,
 ['21du',
  '7e40',
  '7f8p',
  '7u4k',
  '7v39',
  '8cgw',
  '8qtk',
  '8y6w',
  '9dol',
  '9o2m',
  '9prs'])

# Sequences/PSSMs

In [17]:
for pdb in remaining:
    path = Path("ncbi_psiblast_pssms") / pdb
    path.mkdir(exist_ok = True)

    feats = extras_featuresd[pdb]
    chain = feats[('Residues', 'auth_asym_id')].unique().item()
    name = f"{pdb}_{chain}"


    fastaf = path / f"{pdb}.fasta"
    if not fastaf.exists():
        seq = (
            pd.DataFrame(
                Cif(
                    pdb, 
                    filename = Path("..") / "structures" / f"{pdb.lower()}.cif"
                ).cif.data["_entity_poly"], dtype=str
            )
            .query(f"entity_id == '{feats[('Residues', 'label_entity_id')].unique().item()}'")
            ["pdbx_seq_one_letter_code_can"].item()
            .replace("\n", "")
        )
        
        with open(fastaf, "w") as f:
            f.write(f">{name}\n")
            f.write(seq)
        print(fastaf)

    pssmf = path / f"{name}.pssm"
    if not (pssmf).exists():
        for f in path.glob("*.asn"):
            subprocess.run(f"""
psiblast \
-in_pssm {f} \
-subject {fastaf} \
-num_iterations 1 \
-out_ascii_pssm {pssmf} \
-out {fastaf.with_suffix(".out")}""",
                shell=True, check=True
            )
            print(pssmf)

ncbi_psiblast_pssms/9prs/9prs_A.pssm


# Make predictions

Edits throughout to fix:
- Hardcoded paths
- Pass locations of ProtT5 and the nr database
- Use existing .pssm

In [18]:
for pdb, feats in extras_featuresd.items():
    # if pdb == "8aq6": continue
    path = Path("AlloFusion/Case Study").resolve()
    path.mkdir(exist_ok = True)
    try:
        outdir = Path(pdb)
        if not outdir.exists():
            chain = feats[('Residues', 'auth_asym_id')].unique().item()
            
            origpdbf = path.parents[2] / "structures" / f"{pdb.lower()}.pdb"
    
            seq = (
                pd.DataFrame(
                    Cif(pdb, origpdbf.with_suffix(".cif")).cif.data["_entity_poly"], dtype=str
                )
                .query(f"entity_id == '{feats[('Residues', 'label_entity_id')].unique().item()}'")
                ["pdbx_seq_one_letter_code_can"].item()
                .replace("\n", "")
            )

            pssmf = Path("ncbi_psiblast_pssms") / pdb / f"{pdb}_{chain}.pssm"
            if pssmf.exists():
                print(pdb)
                (path / f"{pdb}_{chain}.pssm").symlink_to(pssmf.resolve())
            else:
                print(pdb, "missing pssm")
                continue
            
            pdbf = path / f"{pdb}.pdb"
            if not pdbf.exists():
                pdbf.symlink_to(origpdbf)

            subprocess.run(f"python AlloFusionMain.py --PDBID {pdb} --CHAIN {chain} --SEQ {seq} --huggingface_dir /data/fnerin/huggingface --nr_database /data/fnerin/nr_database/nr", cwd="AlloFusion", shell=True, check=True)

            path.rename(outdir)
            
    except Exception as e:
        print(f"ERROR: ", pdb)
        print(e)
        # path.rename(f"{pdb}_error")
        continue

9prs


2026-05-04 19:20:58.223136: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-04 19:20:58.239199: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Embedding done!
PSSM done!
Bio done!
11/19 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 

19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step


# Process

In [19]:
results = {}

for pdb, feats in extras_featuresd.items():
    # if pdb == "8aq6": continue
    chain = feats[('Residues', 'auth_asym_id')].unique().item()
    resf = f"{pdb}/{pdb}_allosteric_residues.txt"
    if os.path.isfile(resf):
        with open(resf) as f:
            txt = f.read()
        chain = txt.split("Chain", 1)[1].strip().split()[0]
        resids = [x for x in txt.split("resid", 1)[1].replace("(", "").replace(")", "").replace(",", " ").split() if x.isdigit()]   

        results[pdb.lower()] = {"pocket": {"residues": (
            pd.DataFrame({"auth_asym_id": [chain]*len(resids), "auth_seq_id": resids}, dtype=str)
            .merge(Cif(pdb, f"../structures/{pdb}.cif").residues)
            [["auth_asym_id", "auth_seq_id"]]
        )}}

len(results), results

(30,
 {'21du': {'pocket': {'residues':   auth_asym_id auth_seq_id
    0            R         937
    1            R         971
    2            R         974
    3            R        1080
    4            R        1081
    5            R        1099}},
  '22mj': {'pocket': {'residues':   auth_asym_id auth_seq_id
    0            A          87
    1            A          93
    2            A          96
    3            A         118
    4            A         124
    5            A         268
    6            A         304}},
  '6s3a': {'pocket': {'residues':   auth_asym_id auth_seq_id
    0            A         132
    1            A         135
    2            A         136
    3            A         176
    4            A         177
    5            A         190
    6            A         218
    7            A         221}},
  '6vvq': {'pocket': {'residues':    auth_asym_id auth_seq_id
    0             C         426
    1             C         428
    2             C       

In [20]:
pd.to_pickle(results, "allofusion_results.pkl")